# Plus vous payez cher, moins vous gagnez · *The more you pay, the less you earn*

Notebook compagnon du chapitre **20. La croissance déjà anticipée : pourquoi elle est souvent dans les cours** — [lire l'article](https://nmlab.io/ressources/croissance-deja-anticipee-dans-les-cours).
Companion notebook to chapter **20. Growth Already Anticipated: Why It's Often Already in the Price** — [read the article](https://nmlab.io/en/ressources/growth-already-priced-in).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère à partir des **données de Robert Shiller (CAPE)**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from **Robert Shiller's data (CAPE)**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import io
import urllib.request
import pandas as pd
from pandas import DataFrame

SHILLER_URL = "http://www.econ.yale.edu/~shiller/data/ie_data.xls"

def load_shiller() -> DataFrame:
    """Données historiques de Robert Shiller (ie_data.xls) : CAPE mensuel et rendement
    réel des actions sur 10 ans (colonnes précalculées par Shiller), depuis 1871.
    Robert Shiller's historical data: monthly CAPE and the 10-year real stock return."""
    try:                                              # moteur .xls (préinstallé sur Colab)
        import xlrd  # noqa: F401
    except ModuleNotFoundError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd"], check=True)
    raw = urllib.request.urlopen(SHILLER_URL, timeout=60).read()
    sheet = pd.read_excel(io.BytesIO(raw), sheet_name="Data", header=7, engine="xlrd")
    df = pd.DataFrame({
        "year":  pd.to_numeric(sheet.iloc[:, 5], errors="coerce"),          # année décimale
        "cape":  pd.to_numeric(sheet.iloc[:, 12], errors="coerce"),         # CAPE (P/E10)
        "ret10": pd.to_numeric(sheet.iloc[:, 19], errors="coerce") * 100,   # rendement réel 10 ans, %
    })
    return df.dropna(subset=["year"])

shiller = load_shiller()


import numpy as np
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Plus vous payez cher, moins vous gagnez",
        sub="Valorisation de départ et rendement réel des 10 années suivantes, États-Unis, 1881-2024",
        xlab="CAPE au départ (valorisation)",
        ylab="rendement réel des 10 années suivantes, %/an",
        today="aujourd'hui\n≈ 35-40",
        expect="rendement réel attendu\n≈ 0 %/an",
        corr="corrélation ≈ −0,5",
        note="Chaque point : un mois depuis 1881. La valorisation de départ « explique » environ un quart du rendement\n"
             "à dix ans — modeste, mais le signe est sans appel. Source : données de Robert Shiller (CAPE)."),
    "en": dict(
        title="The more you pay, the less you earn",
        sub="Starting valuation and real return over the next 10 years, United States, 1881-2024",
        xlab="starting CAPE (valuation)",
        ylab="real return over the next 10 years, %/yr",
        today="today\n≈ 35-40",
        expect="expected real return\n≈ 0%/yr",
        corr="correlation ≈ −0.5",
        note="Each dot: one month since 1881. The starting valuation « explains » about a quarter of the ten-year\n"
             "return — modest, but the sign is unmistakable. Source: Robert Shiller's data (CAPE)."),
}

def build_figure(shiller: "DataFrame", lang: str) -> Figure:
    """Nuage CAPE de départ vs rendement réel à 10 ans, avec droite de régression."""
    text = LABELS[lang]
    pts = shiller.dropna(subset=["cape", "ret10"])
    x, y = pts["cape"].to_numpy(), pts["ret10"].to_numpy()

    fig = nm.figure(height_px=1083)
    ax = nm.axes(fig, left=0.078, bottom=0.155)
    ax.axvspan(35, 41, color=nm.COLORS["rose"], alpha=0.12, linewidth=0)
    ax.axhline(0, color=nm.COLORS["blue2"], linewidth=1.6, alpha=0.85)
    ax.scatter(x, y, s=12, color=nm.COLORS["blue"], alpha=0.32, linewidths=0, zorder=2)
    slope, intercept = np.polyfit(x, y, 1)
    line_x = np.array([5, 45])
    ax.plot(line_x, intercept + slope * line_x, color=nm.COLORS["amber"], linewidth=4.2, zorder=3)
    ax.set_xlim(4, 46)
    ax.set_xticks(range(5, 46, 5))
    ax.set_ylim(-11, 21)
    ax.set_yticks(range(-10, 21, 5))
    ax.set_xlabel(text["xlab"])
    ax.set_ylabel(text["ylab"])
    ax.text(38, 16.6, text["today"], ha="center", va="center", fontsize=22,
            fontweight="bold", color=nm.COLORS["rose"], linespacing=1.5)
    ax.annotate(text["expect"], xy=(37, intercept + slope * 37), xytext=(30.5, -6.6),
                ha="center", va="center", fontsize=21, color=nm.COLORS["rose"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["rose"], lw=1.8))
    ax.text(13, -7.3, text["corr"], ha="center", va="center", fontsize=25,
            fontweight="bold", color=nm.COLORS["text"])
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(shiller, LANG)